# 01 EDA — PathSnap GPS Map-Matching Dataset
**Person:** P4 | **Course:** UE24CS352A Machine Learning Hackathon

Covers: trips per partition, GPS observations, time gaps, road-network size, reference route quality, and interactive Folium map.
Findings are logged in `docs/data_quality_log.md`.

> **Note:** `results/eda/dataset_inventory.csv` was generated by running `eda/01_inventory_eda.py` on `data/train` and `data/student_test`. The evaluator partition was **never accessed**.

In [ ]:
# --- Imports ---
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from pathsnap.config import load_config
cfg = load_config()

# Load pre-generated EDA outputs
inv  = pd.read_csv('../results/eda/dataset_inventory.csv')
summ = pd.read_csv('../results/eda/summary_stats.csv', index_col=0)

print(f'Inventory loaded: {len(inv)} trips')
print(f'Partitions: {inv.partition.value_counts().to_dict()}')

## 1. Dataset Inventory
Total: **90 trips** — 80 train, 10 student_test. Evaluator test data was never touched.

In [ ]:
# --- Cell 1: Dataset Inventory ---
summary = inv.groupby('partition').agg(
    trips        = ('trip',       'count'),
    gps_min      = ('n_gps',      'min'),
    gps_mean     = ('n_gps',      'mean'),
    gps_max      = ('n_gps',      'max'),
    nodes_mean   = ('n_nodes',    'mean'),
    arcs_mean    = ('n_arcs',     'mean'),
    route_mean   = ('n_route_arcs','mean'),
).round(0).astype(int)

print('=== Dataset Summary ===')
display(summary)

print('\n=== Per-trip inventory (first 5 rows) ===')
display(inv.head())

## 2. Missing-Value Analysis
Expected finding: **zero** missing cells, duplicate rows, and non-monotonic timestamps.

In [ ]:
# --- Cell 2: Missing-Value Analysis ---
total_missing    = inv.missing_cells.sum()
total_dups       = inv.dup_rows.sum()
total_non_mono   = inv.ts_non_monotonic.sum()
lon_invalid      = (~inv.lon_ok).sum()
lat_invalid      = (~inv.lat_ok).sum()
route_invalid    = (~inv.route_ids_valid).sum()
arcs_invalid     = (~inv.arcs_node_ids_valid).sum()
isolated         = inv.isolated_nodes.sum()

quality = pd.DataFrame({
    'Check': [
        'Missing cells (lon/lat/ts)',
        'Duplicate rows',
        'Non-monotonic timestamps',
        'Invalid lon values (outside ±180)',
        'Invalid lat values (outside ±90)',
        'Invalid route arc IDs',
        'Invalid arc node IDs',
        'Isolated nodes (total)',
    ],
    'Count': [
        total_missing, total_dups, total_non_mono,
        lon_invalid, lat_invalid,
        route_invalid, arcs_invalid, isolated
    ],
    'Status': ['✅ Clean' if v == 0 else '⚠️ Check' for v in [
        total_missing, total_dups, total_non_mono,
        lon_invalid, lat_invalid,
        route_invalid, arcs_invalid, isolated
    ]]
})

print('=== Data Quality Checks ===')
display(quality)

## 3. GPS Trajectory Statistics
Histogram of GPS point counts per trip. Note the long-tail trip `00000021` with 15,783 points.

In [ ]:
# --- Cell 3: GPS Trajectory Statistics ---
fig, ax = plt.subplots(figsize=(9, 4))
for part, color in [('train', 'steelblue'), ('student_test', 'darkorange')]:
    s = inv[inv.partition == part]
    ax.hist(s.n_gps, bins=20, alpha=0.6, label=part, color=color)

ax.set_xlabel('GPS points per trip')
ax.set_ylabel('Trip count')
ax.set_title('Distribution of GPS Points per Trip')
ax.legend()
ax.axvline(inv.n_gps.mean(), color='red', linestyle='--', label=f'Mean = {inv.n_gps.mean():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/eda_distributions.png', dpi=150)
plt.show()

print(inv.groupby('partition')['n_gps'].describe().round(1))

## 4. Time-Gap Analysis
97% of time steps are exactly 1 s apart (1 Hz GPS). 4 train trips have gaps exceeding 1,000 s — these are flagged for P3's HMM gap handling.

In [ ]:
# --- Cell 4: Time-Gap Analysis ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for part, color in [('train', 'steelblue'), ('student_test', 'darkorange')]:
    s = inv[inv.partition == part]
    axes[0].hist(np.log10(s.dt_max.clip(lower=1)), bins=20, alpha=0.6, label=part, color=color)
    axes[1].hist(s.n_gaps_gt_10s, bins=20, alpha=0.6, label=part, color=color)

axes[0].set_xlabel('log10(max time gap, s)')
axes[0].set_title('Max Time Gap per Trip (log scale)')
axes[0].legend()

axes[1].set_xlabel('Number of gaps > 10 s')
axes[1].set_title('Gaps > 10 s per Trip')
axes[1].legend()

plt.tight_layout()
plt.show()

# Anomalous trips
print('=== Anomalous trips with large gaps ===')
anomalous = inv[inv.dt_max > 1000][['partition', 'trip', 'n_gps', 'dt_max', 'n_gaps_gt_10s']].sort_values('dt_max', ascending=False)
display(anomalous)

## 5. Road-Network Statistics
Each trip has its own local road network. Arc counts vary widely — from 649 to 307,392 per trip.

In [ ]:
# --- Cell 5: Road-Network Statistics ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for part, color in [('train', 'steelblue'), ('student_test', 'darkorange')]:
    s = inv[inv.partition == part]
    axes[0].hist(np.log10(s.n_arcs),  bins=20, alpha=0.6, label=part, color=color)
    axes[1].hist(np.log10(s.n_nodes), bins=20, alpha=0.6, label=part, color=color)

axes[0].set_xlabel('log10(arcs per trip network)')
axes[0].set_title('Road Network Arc Count (log scale)')
axes[0].legend()

axes[1].set_xlabel('log10(nodes per trip network)')
axes[1].set_title('Road Network Node Count (log scale)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(inv.groupby('partition')[['n_nodes', 'n_arcs']].describe().round(0).astype(int))

## 6. Reference-Route Quality
11 trips have at least one route break (arc _i_ does not chain to arc _i+1_). This is a known annotation artefact — not a pipeline error.

In [ ]:
# --- Cell 6: Reference-Route Quality ---
broken = inv[inv.ref_route_breaks > 0][['partition', 'trip', 'n_route_arcs', 'ref_route_breaks']].sort_values('ref_route_breaks', ascending=False)

print(f'Trips with route breaks: {len(broken)} / {len(inv)}')
print(f'  Train:        {len(broken[broken.partition=="train"])}')
print(f'  Student test: {len(broken[broken.partition=="student_test"])}')
print()
display(broken)

print()
print('A route break means arc[i].v != arc[i+1].u — the reference arcs do not chain continuously.')
print('This is a label quality issue, not a prediction error. These trips are retained for training.')

## 7. Folium Interactive Map
GPS points (blue), reference route (green), road network (grey) for trip `00000046`.

In [ ]:
# --- Cell 7: Folium Interactive Map ---
from pathsnap.viz.folium_maps import make_trip_map
import IPython.display as ipd

trip_dir = '../data/train/00000046'
m = make_trip_map(trip_dir, max_net_arcs=3000)

# Save for reports
m.save('../reports/maps/train_00000046.html')
print('Map saved to reports/maps/train_00000046.html')

# Show inline in notebook
ipd.display(ipd.HTML(m._repr_html_()))

## 8. Assumptions and Decisions
*(Mirrored in `docs/data_quality_log.md`)*

In [ ]:
# --- Cell 8: Assumptions and Decisions ---
assumptions = [
    'Evaluator partition (data/evaluator_test) was NEVER accessed during EDA.',
    'Only train and student_test partitions used for all analysis.',
    'Dataset is clean: 0 missing cells, 0 duplicate rows, 0 non-monotonic timestamps.',
    'GPS sampling is nominally 1 Hz — 97% of time steps are exactly 1 s apart.',
    '4 anomalous train trips have GPS gaps > 1000 s — flagged to P3 for HMM gap handling.',
    '11 trips have reference route breaks — treated as label noise, not pipeline errors.',
    'Road network is per-trip local — coordinate projection must be done per trip (not globally).',
    'GPS coordinates span multiple countries (lon -100 to 137, lat -35 to 62).',
    'Folium maps capped at max_net_arcs=3000 to keep HTML file size manageable.',
]

print('=== Key Assumptions and Decisions ===')
for i, a in enumerate(assumptions, 1):
    print(f'{i}. {a}')